# Passport Seva RAG - Evaluation Notebook

**SMAI Assignment 3 · T10.1 Passport Assistant (Tier 1)**

_This is a **retrieval / evaluation** notebook - there is **no model training** (pre-trained embeddings + RAG only)._

This notebook supports **qualitative evaluation** of the RAG pipeline:
- **Retrieval testing** (which PDFs/pages come back for each question?)
- **Answer quality** checklist (after running the Streamlit app or calling `generate_answer`)
- **Citation** checklist (filename + page shown and plausible)

**Prerequisite:** Run `python ingest.py` from the project root so `chroma_db/` is populated.

**Tip:** Run Jupyter from the project root, or adjust `PROJECT_ROOT` in the next cell.


## 1. Environment setup


In [ ]:
import sys
from pathlib import Path

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "rag.py").is_file():
    pass
elif (PROJECT_ROOT.parent / "rag.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError(
        "Run this notebook from `passport-rag-chatbot/` or `passport-rag-chatbot/notebooks/`."
    )

sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())


In [ ]:
import os

# Load .env from project root so GEMINI_API_KEY is available if you test generation here
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

from rag import retrieve_context, generate_answer, vector_store_document_count
from config import TOP_K

print("Chroma chunk count:", vector_store_document_count())
print("TOP_K:", TOP_K)


## 2. Sample evaluation questions

Use the same set when comparing **ablations** (e.g. different `TOP_K` or `CHUNK_SIZE` in `config.py` - re-run `ingest.py` after chunk size changes).


In [ ]:
SAMPLE_QUESTIONS = [
    "How do I apply for a new passport?",
    "How do I book a passport appointment?",
    "How do I check my application status?",
    "What is Tatkaal passport?",
    "What happens at Passport Seva Kendra?",
    "How do I fill the online passport application form?",
]

SAMPLE_QUESTIONS


## 3. Retrieval testing

For each question, inspect **top retrieved chunks**: `source` (PDF filename), **page**, and a **text preview**. If the wrong booklet dominates, add better PDFs or tune `TOP_K` / chunking.


In [ ]:
def print_retrieval_report(question: str, top_k: int | None = None) -> None:
    """Pretty-print top-k retrieval for one question."""
    kwargs = {}
    if top_k is not None:
        kwargs["top_k"] = top_k
    try:
        ctxs = retrieve_context(question, **kwargs)
    except Exception as e:
        print("Retrieval error:", e)
        return
    print("Question:", question)
    print("Hits:", len(ctxs))
    for i, c in enumerate(ctxs, start=1):
        preview = (c["text"] or "")[:320].replace("\n", " ")
        print(
            f"  [{i}] {c['source']}  p.{c['page']}  chunk#{c.get('chunk_index', 0)}\n"
            f"      {preview}..."
        )
    print()


for q in SAMPLE_QUESTIONS:
    print_retrieval_report(q)


## 4. Optional: one end-to-end answer (Gemini)

Requires valid `GEMINI_API_KEY` in `.env`. Skip this cell in class if API usage is restricted.


In [ ]:
if not os.getenv("GEMINI_API_KEY"):
    print("Skip: GEMINI_API_KEY not set.")
else:
    q = SAMPLE_QUESTIONS[0]
    answer, cites, err = generate_answer(q, hindi=False)
    print("Q:", q)
    if err:
        print("Error:", err)
    print("\n--- Answer ---\n", answer)
    print("\nCitations:", cites)


## 5. Answer quality checklist (manual)

For each trial question, after you read the assistant reply in Streamlit (or above), mark:

- [ ] Answer **only** restates or paraphrases **retrieved** content (no invented fees/lists).
- [ ] If PDFs lack the fact, the bot says **"I could not find this in the official documents I have."**
- [ ] User is reminded to check the **official Passport Seva** portal for latest updates.
- [ ] Structure follows: direct answer → steps → documents → notes → sources (as applicable).


## 6. Citation correctness checklist (manual)

- [ ] Each shown citation has a **real** filename from `data/pdfs/`.
- [ ] Page numbers are **plausible** when you open the PDF (same section/topic).
- [ ] Duplicate (file, page) pairs are **not** repeated in the UI list.


## 7. Result table (fill after your evaluation)

Columns: **question**, **expected topic**, **answer correctness**, **citation shown**, **notes**.


In [ ]:
import pandas as pd

rows = [
    {
        "question": "How do I apply for a new passport?",
        "expected_topic": "Application flow / steps",
        "answer_correctness": "",
        "citation_shown": "",
        "notes": "",
    },
    {
        "question": "How do I book a passport appointment?",
        "expected_topic": "Appointment booking",
        "answer_correctness": "",
        "citation_shown": "",
        "notes": "",
    },
    {
        "question": "How do I check my application status?",
        "expected_topic": "Status tracking",
        "answer_correctness": "",
        "citation_shown": "",
        "notes": "",
    },
    {
        "question": "What is Tatkaal passport?",
        "expected_topic": "Tatkaal scheme / undertaking",
        "answer_correctness": "",
        "citation_shown": "",
        "notes": "",
    },
    {
        "question": "What happens at Passport Seva Kendra?",
        "expected_topic": "PSK visit / steps at Kendra",
        "answer_correctness": "",
        "citation_shown": "",
        "notes": "",
    },
    {
        "question": "How do I fill the online passport application form?",
        "expected_topic": "Online form / fields",
        "answer_correctness": "",
        "citation_shown": "",
        "notes": "",
    },
]

eval_df = pd.DataFrame(rows)
eval_df


## 8. Simple visualization (optional)

After filling **answer correctness** as `1` (good) / `0` (poor), you can plot a quick bar chart. Below uses placeholder scores.


In [ ]:
import matplotlib.pyplot as plt

# Example: replace with your numeric scores (0–1) per question index
scores = [1, 1, 1, 0.5, 1, 0.5]
labels = [f"Q{i+1}" for i in range(len(scores))]

plt.figure(figsize=(8, 3))
plt.bar(labels, scores, color="#2E86AB")
plt.ylim(0, 1.05)
plt.ylabel("Manual QA score (0–1)")
plt.title("Passport Seva RAG - sample question scores")
plt.tight_layout()
plt.show()


## 9. Ablation hooks

- Change **`TOP_K`** in `config.py` → restart kernel → re-run retrieval cells.
- Change **`CHUNK_SIZE`** → run `python ingest.py` again → reload this notebook.
- Compare **English** vs **Hindi** using `generate_answer(q, hindi=True)`.
